<div dir="rtl" align="right">

# مُرشِّحُ المتوسطِ المتحرّكِ \(Moving Average Filter\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يُحسبُ مُرشِّحُ المتوسطِ المتحرّكِ قيمةَ كلِّ عيّنةٍ كمتوسطِ عيّناتٍ مُجاورةٍ لها. كلّما كَبُرَ حجمُ النافذةِ، زادَ التنعيمُ واختفى الضجيجُ، لكنّ الإشارةَ تَفقدُ تفاصيلَها الدقيقةَ.

## المُخرجاتُ المُتوقّعةُ

- النافذةُ الصغيرةُ (5): تَنعيمٌ خفيفٌ، يَبقى الضجيجُ واضحًا
- النافذةُ المتوسطةُ (11): تَنعيمٌ مُعتدلٌ، تَختفي التفاصيلُ الدقيقةُ
- النافذةُ الكبيرةُ (21): تَنعيمٌ قويٌّ، تَصبحُ الإشارةُ ناعمةً جدًا وتَفقدُ الشوائبَ السريعةَ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| معدّلُ الأخذِ | 200 Hz | عيّنةٌ كلَّ 5 ms |
| النوافذُ | 5, 11, 21 | أحجامُ النافذةِ |
| العيّناتُ المرسومةُ | 5000 | أولُ 25 ثانيةً |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ المُرشِّحِ

نَستخدمُ `np.convolve` مع `mode='same'` لتطبيقِ المتوسطِ المتحرّكِ. يَكُونُ النواةُ متجهًا من الواحداتِ مُقسومًا على حجمِ النافذةِ.

</div>

In [ ]:
windows = [5, 11, 21]
filtered = {}
for w in windows:
    kernel = np.ones(w) / w
    filtered[w] = np.convolve(channel_data, kernel, mode='same')
print(f'Applied moving average with windows: {windows}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- النافذةُ 5: يَبقى الضجيجُ واضحًا
- النافذةُ 11: يَختفي الضجيجُ السريعُ وتَبقى التفاصيلُ
- النافذةُ 21: تَنعيمٌ قويٌّ يُخفي الشوائبَ السريعةَ
- استخدمْ أداةَ التكبيرِ لفحصِ نطاقاتَ زمنيةٍ مُحدّدةٍ

</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=('Original (P4)',
                                    'Moving average (window=5)',
                                    'Moving average (window=11)',
                                    'Moving average (window=21)'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Raw',
                         line=dict(color='gray', width=0.5)), row=1, col=1)
for i, w in enumerate(windows, start=2):
    fig.add_trace(go.Scatter(x=t_sec, y=filtered[w][:n_plot],
                             name=f'w={w}', line=dict(width=0.5)), row=i, col=1)
fig.update_layout(height=900, title_text='Moving Average Filter - Channel P4',
                  xaxis4_title='Time (s)', showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- يُقلّلُ المتوسطُ المتحرّكُ الضجيجَ بمتوسطِ العيّناتِ المُجاورةِ
- النوافذُ الكبيرةُ تُنتجُ تنعيمًا أقوى لكنّها تُخفي التفاصيلَ
- النوافذُ الصغيرةُ تُحافظُ على التفاصيلِ لكنّها لا تُزيلُ الضجيجَ تمامًا
- يَختارُ الباحثُ حجمَ النافذةِ وفقًا للموازنةِ بين التنعيمِ والحفاظِ على التفاصيلِ

</div>